# 04 · The same numbers in SQL

**Produces:** the answers to our key questions from a database, a comparison with pandas, and `sql/06_brief_a.sql` (the queries a finance analyst can re-run).

**Needs:** notebook 03 run.

**Why SQL after pandas?** The warehouse is where finance will re-run our numbers. If SQL and pandas disagree, one of them has a bug, and it is better that we find it than the panel. We use SQLite because it comes with Python, so nothing needs installing.

## CONFIG
Everything that depends on the machine or the period lives in this one cell, so nothing else needs editing.

In [ ]:
# ---------- imports ----------
import gzip
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# show more columns when we print a dataframe
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

# ---------- paths ----------
# the notebooks live in the notebooks folder, so the repo root is one folder up
if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

RAW = ROOT / "data" / "raw"       # the data pack as we received it (we never edit this)
CLEAN = ROOT / "data" / "clean"   # the files our notebooks write
SLIDES = ROOT / "slides"          # charts and the excel file for the slides
CLEAN.mkdir(parents=True, exist_ok=True)
SLIDES.mkdir(parents=True, exist_ok=True)

# ---------- the two periods we compare ----------
H1_2025 = ("2025-01-01", "2025-06-30 23:59:59")   # 23:59:59 so that 30 June is included
H1_2026 = ("2026-01-01", "2026-06-30 23:59:59")

# a small helper: prints PASS or FAIL but never stops the notebook
def check(name, condition, detail=""):
    if condition:
        print("PASS", name, detail)
    else:
        print("FAIL", name, detail)

print("repo root:", ROOT)

In [ ]:
import sqlite3
print("sqlite version:", sqlite3.sqlite_version)

# ---------- load the clean table (only the columns we need in SQL) ----------
cols = ["booking_id", "city", "channel", "source", "slot_at", "person_id", "test_price"]
df = pd.read_csv(CLEAN / "bookings_clean.csv.gz", usecols=cols, parse_dates=["slot_at"],
                 dtype={"booking_id": "str", "person_id": "str"}, low_memory=False)
print(df.shape)

# SQLite has no date type, so we store the date as text like 2026-04-25 10:16:00 (this sorts correctly)
df["slot_at"] = df["slot_at"].dt.strftime("%Y-%m-%d %H:%M:%S")
df.head()

## 1 · DO: load the tables into a database

In [ ]:
# an in-memory database: it disappears when the notebook stops, so nothing big is added to the repo
con = sqlite3.connect(":memory:")

df.to_sql("bookings", con, index=False, chunksize=100000)

# the plan: from wide (one column per month) to long (one row per city and month)
plan_wide = pd.read_csv(RAW / "plan" / "plan_2026_H1.csv")
plan_long = plan_wide.melt(id_vars=["city", "metric"], var_name="month", value_name="plan_tests")
plan_long["city"] = plan_long["city"].replace({"Bangalore": "Bengaluru"})
plan_long[["city", "month", "plan_tests"]].to_sql("plan", con, index=False)

def run_sql(query):
    # run a query and give back a dataframe
    return pd.read_sql_query(query, con)

print(run_sql("SELECT COUNT(*) AS rows_in_bookings FROM bookings"))
print(run_sql("SELECT COUNT(*) AS rows_in_plan FROM plan"))

## 2 · The queries
Every query says what question it answers, the **grain** (what one row is) and the **denominator** in a comment. This is the rule for anything we hand to finance.

In [ ]:
# ---------- Query 1: the headline ----------
q1 = """
-- Question: how many tests in each half-year, and what is the growth?
-- Grain: one row for the whole network.   Denominator: tests in H1 2025.
SELECT
    SUM(CASE WHEN slot_at >= '2025-01-01' AND slot_at < '2025-07-01' THEN 1 ELSE 0 END) AS tests_h1_2025,
    SUM(CASE WHEN slot_at >= '2026-01-01' AND slot_at < '2026-07-01' THEN 1 ELSE 0 END) AS tests_h1_2026
FROM bookings
"""
r1 = run_sql(q1)
print(r1)
sql_h1_2025 = int(r1["tests_h1_2025"][0])
sql_h1_2026 = int(r1["tests_h1_2026"][0])
print("growth from SQL:", round((sql_h1_2026 / sql_h1_2025 - 1) * 100, 2), "%")

In [ ]:
# ---------- Query 2: actual against plan, by city ----------
q2 = """
-- Question: how far is each city from its plan, in points of the network's H1 2025 tests?
-- Grain: one row per city.   Denominator: network tests in H1 2025 (the six gap_points add up to the network gap).
WITH actual AS (
    SELECT city,
           SUM(CASE WHEN slot_at >= '2025-01-01' AND slot_at < '2025-07-01' THEN 1 ELSE 0 END) AS tests_2025,
           SUM(CASE WHEN slot_at >= '2026-01-01' AND slot_at < '2026-07-01' THEN 1 ELSE 0 END) AS tests_2026
    FROM bookings
    GROUP BY city
),
plan_city AS (
    SELECT city, SUM(plan_tests) AS plan_2026
    FROM plan
    GROUP BY city
),
total AS (
    SELECT SUM(tests_2025) AS total_2025 FROM actual
)
SELECT a.city, a.tests_2025, a.tests_2026, p.plan_2026,
       ROUND(100.0 * (a.tests_2026 - a.tests_2025) / a.tests_2025, 1) AS growth_pct,
       ROUND(100.0 * (a.tests_2026 - p.plan_2026) / t.total_2025, 2) AS gap_points
FROM actual a
JOIN plan_city p ON p.city = a.city
CROSS JOIN total t
ORDER BY gap_points
"""
r2 = run_sql(q2)
print(r2.to_string(index=False))
print("sum of gap points:", round(r2["gap_points"].sum(), 2))

In [ ]:
# ---------- Query 3: the tree (tests, visits, people) ----------
q3 = """
-- Question: tests = people x visits per person x tests per visit. what are the three numbers in each half?
-- Grain: one row per half-year.   Denominator: shown in each column name.
SELECT half,
       COUNT(*)                      AS tests,
       COUNT(DISTINCT booking_id)    AS visits,
       COUNT(DISTINCT person_id)     AS people
FROM (
    SELECT booking_id, person_id,
           CASE WHEN slot_at >= '2025-01-01' AND slot_at < '2025-07-01' THEN 'H1_2025'
                WHEN slot_at >= '2026-01-01' AND slot_at < '2026-07-01' THEN 'H1_2026' END AS half
    FROM bookings
)
WHERE half IS NOT NULL
GROUP BY half
ORDER BY half
"""
r3 = run_sql(q3)
r3["visits_per_person"] = (r3["visits"] / r3["people"]).round(3)
r3["tests_per_visit"] = (r3["tests"] / r3["visits"]).round(3)
print(r3.to_string(index=False))

In [ ]:
# ---------- Query 4: clinic against home ----------
q4 = """
-- Question: how did clinic and home bookings change between the two halves?
-- Grain: one row per channel.   Denominator: that channel's tests in H1 2025.
SELECT channel,
       SUM(CASE WHEN slot_at >= '2025-01-01' AND slot_at < '2025-07-01' THEN 1 ELSE 0 END) AS tests_2025,
       SUM(CASE WHEN slot_at >= '2026-01-01' AND slot_at < '2026-07-01' THEN 1 ELSE 0 END) AS tests_2026
FROM bookings
GROUP BY channel
"""
r4 = run_sql(q4)
r4["growth_pct"] = ((r4["tests_2026"] / r4["tests_2025"] - 1) * 100).round(1)
print(r4.to_string(index=False))

In [ ]:
# ---------- Query 5: month by month, compared with the same month last year (a window function) ----------
q5 = """
-- Question: for each city and month of 2026, how does it compare with the same month of 2025?
-- Grain: one row per city per month.   Denominator: the same city and month one year earlier.
WITH monthly AS (
    SELECT city, substr(slot_at, 1, 7) AS month, COUNT(*) AS tests
    FROM bookings
    GROUP BY city, substr(slot_at, 1, 7)
),
with_last_year AS (
    SELECT city, month, tests,
           LAG(tests, 12) OVER (PARTITION BY city ORDER BY month) AS same_month_last_year
    FROM monthly
)
SELECT city, month, tests, same_month_last_year,
       ROUND(100.0 * (tests - same_month_last_year) / same_month_last_year, 1) AS yoy_pct
FROM with_last_year
WHERE month >= '2026-01' AND month <= '2026-06'
ORDER BY city, month
"""
r5 = run_sql(q5)
print(r5.pivot(index="month", columns="city", values="yoy_pct"))

> **Interview angle.** In what order does the database run a SELECT with a GROUP BY and a HAVING? Use it to explain why a column alias from SELECT cannot be used in WHERE.
>
> *A good answer:* FROM and JOIN, then WHERE, then GROUP BY, then HAVING, then SELECT, then ORDER BY and LIMIT. WHERE runs before SELECT has made the alias, so the alias does not exist yet. HAVING runs after grouping, so it can see totals. ORDER BY runs after SELECT, so it can see aliases.

> **Interview angle.** When does a window function beat a GROUP BY?
>
> *A good answer:* when every row must stay and we need a value computed from its neighbours: the same month last year, a rank inside a city, a running total. Here `LAG(tests, 12)` looks twelve months back inside each city.

## 3 · CHECK: does SQL give the same numbers as pandas?

In [ ]:
# the same numbers computed with pandas, from the same clean table
df_pd = pd.read_csv(CLEAN / "bookings_clean.csv.gz", usecols=["booking_id", "city", "slot_at", "person_id"],
                    parse_dates=["slot_at"], low_memory=False)
pd_h1_2025 = df_pd["slot_at"].between(H1_2025[0], H1_2025[1]).sum()
pd_h1_2026 = df_pd["slot_at"].between(H1_2026[0], H1_2026[1]).sum()

check("H1 2025 total: SQL = pandas", sql_h1_2025 == pd_h1_2025, str(sql_h1_2025) + " vs " + str(pd_h1_2025))
check("H1 2026 total: SQL = pandas", sql_h1_2026 == pd_h1_2026, str(sql_h1_2026) + " vs " + str(pd_h1_2026))

# by city
pd_city_2025 = df_pd[df_pd["slot_at"].between(H1_2025[0], H1_2025[1])].groupby("city").size()
pd_city_2026 = df_pd[df_pd["slot_at"].between(H1_2026[0], H1_2026[1])].groupby("city").size()
same_cities = True
for i in range(len(r2)):
    city = r2["city"][i]
    if r2["tests_2025"][i] != pd_city_2025[city] or r2["tests_2026"][i] != pd_city_2026[city]:
        same_cities = False
check("every city total: SQL = pandas", same_cities)

# people from the tree query against pandas
pd_people_2026 = df_pd[df_pd["slot_at"].between(H1_2026[0], H1_2026[1])]["person_id"].nunique()
sql_people_2026 = int(r3[r3["half"] == "H1_2026"]["people"].iloc[0])
check("people in H1 2026: SQL = pandas", pd_people_2026 == sql_people_2026, str(sql_people_2026) + " vs " + str(pd_people_2026))

## 4 · SUM: save the queries for finance

In [ ]:
# write the queries into one .sql file, so finance can run them without Python
sql_folder = ROOT / "sql"
sql_folder.mkdir(exist_ok=True)

header = ("-- Brief A: queries behind the headline. Written for SQLite.\n"
          "-- Table bookings: one row per booking-test, cleaned (notebook 03). Table plan: one row per city and month.\n\n")
text = header
for q in [q1, q2, q3, q4, q5]:
    text = text + q.strip() + ";\n\n"
open(sql_folder / "06_brief_a.sql", "w").write(text)
print("wrote", sql_folder / "06_brief_a.sql", len(text), "characters")

## Before you move on
- Which of our pandas numbers did we reproduce in SQL, and did they agree to the last test?
- Which query would we hand to finance, and could they read it without us in the room?
- If SQL and pandas had disagreed, which would we check first?